In [1]:
import pandas as pd

team_name_mapping = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Oakland Raiders": "LV",  # Fixed
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Diego Chargers": "LAC",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "St. Louis Rams": "LAR",  # Fixed
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Washington Football Team": "WAS"  # Fixed
}

# Function to replace team names in the DataFrame
def replace_team_names(df, mapping):
    df['Team'] = df['Team'].map(mapping)
    return df

# File paths
files = {
    "Half PPR": "../PickleFiles/Half PPR Rankings.pkl",
    "Full PPR": "../PickleFiles/Full PPR Rankings.pkl",
    "Non PPR": "../PickleFiles/Non PPR Rankings.pkl"
}

# Define weights
vbd_weight = 1
adp_weight = 0

# Loop through each file
for key, file in files.items():
    # Load the pickle file
    df = pd.read_pickle(file)
    
    # Get the baseline points for each position
    baseline_qb = df[(df['Position'] == 'QB')]['Final PPG'].iloc[min(11, len(df[df['Position'] == 'QB'])-1)]
    baseline_rb = df[(df['Position'] == 'RB')]['Final PPG'].iloc[min(23, len(df[df['Position'] == 'RB'])-1)]
    baseline_wr = df[(df['Position'] == 'WR')]['Final PPG'].iloc[min(29, len(df[df['Position'] == 'WR'])-1)]
    baseline_te = df[(df['Position'] == 'TE')]['Final PPG'].iloc[min(11, len(df[df['Position'] == 'TE'])-1)]
    
    # Calculate VBD for each player based on their position
    def calculate_vbd(row):
        if row['Position'] == 'QB':
            return row['Final PPG'] - baseline_qb
        elif row['Position'] == 'RB':
            return row['Final PPG'] - baseline_rb
        elif row['Position'] == 'WR':
            return row['Final PPG'] - baseline_wr
        elif row['Position'] == 'TE':
            return row['Final PPG'] - baseline_te
        else:
            return 0
    
    df['VBD'] = df.apply(calculate_vbd, axis=1)
    
    # Normalize VBD and ESPN ADP
    df['Normalized VBD'] = (df['VBD'] - df['VBD'].min()) / (df['VBD'].max() - df['VBD'].min())
    df['Normalized ADP'] = (df['ESPN ADP'] - df['ESPN ADP'].min()) / (df['ESPN ADP'].max() - df['ESPN ADP'].min())
    
    # Calculate the weighted score
    df['Weighted Score'] = (vbd_weight * df['Normalized VBD']) + (adp_weight * (1 - df['Normalized ADP']))
    
    # Sort by weighted score in descending order to get the top players based on the weighted score
    df = df.sort_values(by='Weighted Score', ascending=False)
    
    # Adjust the ranking column
    df['Rank'] = range(1, len(df) + 1)

    df = replace_team_names(df, team_name_mapping)

    df['Position Rank'] = df.groupby('Position').cumcount() + 1
    df['Position'] = df['Position'] + df['Position Rank'].astype(str)
    df.drop(columns=['Position Rank'], inplace=True)
    
    # Save the updated DataFrame to a new pickle file
    output_file = file.replace(".pkl", " with Weighted VBD.pkl")
    df.to_pickle(output_file)

df

/Users/kmaran3/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Rank,Name,Team,Position,Final PPG,Bye Week,ESPN ADP,VBD,Normalized VBD,Normalized ADP,Weighted Score
24,1,Tyler Allgeier,ATL,RB1,10.983450,12,88.0,5.388146,0.829508,0.350806,0.829508
0,2,Bryce Young,CAR,QB1,17.553673,11,145.0,4.592933,0.767567,0.580645,0.767567
1,3,Jalen Hurts,PHI,QB2,17.299791,5,70.0,4.339051,0.747792,0.278226,0.747792
31,4,Breece Hall,NYJ,RB2,9.733611,12,31.0,4.138307,0.732156,0.120968,0.732156
43,5,Kyren Williams,LAR,RB3,8.763206,6,36.0,3.167902,0.656569,0.141129,0.656569
...,...,...,...,...,...,...,...,...,...,...,...
279,281,C.J. Ham,MIN,RB69,1.227260,6,NaN,-4.368044,0.069577,NaN,NaN
280,282,Dare Ogunbowale,HOU,RB70,1.130285,14,NaN,-4.465019,0.062023,NaN,NaN
281,283,Marcedes Lewis,CHI,TE68,0.966873,7,NaN,-4.890017,0.028919,NaN,NaN
282,284,Mike Williams,PIT,WR105,0.772190,9,NaN,-4.083860,0.091713,NaN,NaN


In [2]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np

# Try 2025 first, fall back to 2024
for year in [2025, 2024]:
    try:
        seasonal = nfl.import_seasonal_data([year])
        comparison_year = year
        break
    except Exception:
        continue

seasonal = seasonal[seasonal['season_type'] == 'REG']

# Get player names
players = nfl.import_players()
player_map = players[['gsis_id', 'display_name', 'position']].drop_duplicates('gsis_id')
actual = seasonal.merge(player_map, left_on='player_id', right_on='gsis_id', how='left')

# Calculate actual PPG for each scoring type
actual['Actual PPG (Non PPR)'] = actual['fantasy_points'] / actual['games']
actual['Actual PPG (Full PPR)'] = actual['fantasy_points_ppr'] / actual['games']
actual['Actual PPG (Half PPR)'] = (actual['fantasy_points'] + actual['receptions'] * 0.5) / actual['games']

# Load predicted rankings
scoring_types = {
    'Full PPR': '../PickleFiles/Full PPR Rankings.pkl',
    'Half PPR': '../PickleFiles/Half PPR Rankings.pkl',
    'Non PPR': '../PickleFiles/Non PPR Rankings.pkl',
}

actual_col_map = {
    'Full PPR': 'Actual PPG (Full PPR)',
    'Half PPR': 'Actual PPG (Half PPR)',
    'Non PPR': 'Actual PPG (Non PPR)',
}

print(f"Comparing predictions vs actual {comparison_year} season stats\n")
print("=" * 70)

for scoring_type, pkl_path in scoring_types.items():
    predicted = pd.read_pickle(pkl_path)[['Name', 'Final PPG']]
    actual_col = actual_col_map[scoring_type]
    actual_subset = actual[['display_name', actual_col]].rename(
        columns={'display_name': 'Name', actual_col: 'Actual PPG'}
    )

    # Merge predicted with actual
    comparison = predicted.merge(actual_subset, on='Name', how='inner')
    comparison['Error'] = comparison['Final PPG'] - comparison['Actual PPG']
    comparison['Abs Error'] = comparison['Error'].abs()

    # Metrics
    mae = comparison['Abs Error'].mean()
    rmse = np.sqrt((comparison['Error'] ** 2).mean())
    corr = comparison['Final PPG'].corr(comparison['Actual PPG'])
    matched = len(comparison)

    print(f"\n{'─' * 70}")
    print(f"  {scoring_type}  |  Matched Players: {matched}")
    print(f"{'─' * 70}")
    print(f"  MAE:  {mae:.2f}  |  RMSE: {rmse:.2f}  |  Correlation: {corr:.3f}")

    # Show top 10 biggest misses
    top_misses = comparison.sort_values('Abs Error', ascending=False).head(10)
    print(f"\n  Top 10 biggest prediction misses:")
    print(top_misses[['Name', 'Final PPG', 'Actual PPG', 'Error']].to_string(index=False))

    # Show top 10 most accurate
    top_accurate = comparison.sort_values('Abs Error').head(10)
    print(f"\n  Top 10 most accurate predictions:")
    print(top_accurate[['Name', 'Final PPG', 'Actual PPG', 'Error']].to_string(index=False))

print(f"\n{'=' * 70}")

Comparing predictions vs actual 2024 season stats


──────────────────────────────────────────────────────────────────────
  Full PPR  |  Matched Players: 279
──────────────────────────────────────────────────────────────────────
  MAE:  4.99  |  RMSE: 6.43  |  Correlation: 0.134

  Top 10 biggest prediction misses:
            Name  Final PPG  Actual PPG      Error
  Saquon Barkley   2.511600   22.206250 -19.694650
   Derrick Henry   2.195004   19.788235 -17.593232
   Ja'Marr Chase   6.888689   23.705882 -16.817194
   Lamar Jackson   8.996358   25.316471 -16.320113
   Luke Musgrave  16.711664    1.642857  15.068807
       Joe Mixon   2.248697   17.178571 -14.929874
Justin Jefferson   3.961369   18.675294 -14.713926
      Josh Allen   9.162870   23.271250 -14.108380
      Sam Howell  12.986404   -0.840000  13.826404
      Mike Evans   3.396588   17.171429 -13.774840

  Top 10 most accurate predictions:
             Name  Final PPG  Actual PPG     Error
    Tommy Tremble   5.665866    5